# Action Clip Bot — Colab GPU backend

**One cell below does everything:** install → server → tunnel URL → keep-alive.
Copy it into Colab (or open this notebook directly), run it, paste the printed
URL into your bot dashboard (**GPU → Default Gpu → Colab → Save**).

> **GPU sizes:** A100 40 GB+ runs the full Wan 2.2 14B. A free **T4 auto-uses
> Wan 2.1 1.3B** (lower fidelity, minutes per clip — but it runs).
> Keep the tab open while generating. Tunnel URLs change every runtime.

In [ ]:
# ============================================================
# Action Clip Bot — Colab GPU backend (SINGLE CELL — just run it)
# Installs deps, starts the video server, opens a public tunnel,
# prints the URL to paste in the dashboard, then keeps the
# runtime alive. Keep this tab open during generation.
# ============================================================

# ---- 0. dependencies (once per runtime, ~3 min) ----
!pip install -q diffusers "transformers==4.57.6" accelerate fastapi uvicorn nest_asyncio httpx bitsandbytes hf_transfer imageio imageio-ffmpeg opencv-python-headless
!curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
!cloudflared --version
!nvidia-smi --query-gpu=name,memory.total --format=csv

# ---- 1. fetch gpu_server.py ----
import os, re, subprocess, time, urllib.request

SERVER_URL = 'https://raw.githubusercontent.com/csjony/action-clip-bot/master/gpu_server.py'
if not os.path.exists('/content/gpu_server.py'):
    print('downloading gpu_server.py from GitHub...')
    try:
        urllib.request.urlretrieve(SERVER_URL, '/content/gpu_server.py')
    except Exception as e:
        print(f'download failed ({e})')
        print('→ upload gpu_server.py manually via the Files panel, then run this cell again.')
assert os.path.exists('/content/gpu_server.py'), 'gpu_server.py missing — upload it via the Files panel.'
os.makedirs('/content/hf_cache', exist_ok=True)
os.environ.setdefault('HF_HOME', '/content/hf_cache')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

# ---- 2. auto-pick a model the GPU can fit ----
# 14B needs 20 GB+ VRAM — a free T4 (15 GB) gets Wan 2.1 1.3B instead
# (lower fidelity, but it runs). Force another with:
#   os.environ['WAN_MODEL_ID'] = 'Wan-AI/Wan2.2-T2V-A14B-Diffusers'
try:
    _q = subprocess.check_output(['nvidia-smi', '--query-gpu=memory.total',
                                  '--format=csv,noheader,nounits'], text=True)
    _vram_mb = int(_q.strip().split()[0])
except Exception:
    _vram_mb = 0
print(f'detected VRAM: {_vram_mb} MiB')
if 'WAN_MODEL_ID' not in os.environ:
    if _vram_mb and _vram_mb < 20000:
        os.environ['WAN_MODEL_ID'] = 'Wan-AI/Wan2.1-T2V-1.3B-Diffusers'
        print('small GPU → Wan 2.1 1.3B (fast, lower fidelity)')
    else:
        print('big GPU → default Wan 2.2 14B (best quality)')
print('WAN_MODEL_ID =', os.environ.get('WAN_MODEL_ID', '(server default 14B)'))

# ---- 3. launch the video server in the background ----
!pkill -f 'gpu_server.py' || true
!nohup python3 -u /content/gpu_server.py > /content/gpu_server.log 2>&1 & echo "server pid: $!"
print('server starting — model download + load takes a while on first boot.')

# ---- 4. open a public tunnel (cloudflared, else localtunnel fallback) ----
def _wait_url(logpath, pattern, exclude=None, rounds=45):
    for _ in range(rounds):
        time.sleep(2)
        try:
            log = open(logpath).read()
        except FileNotFoundError:
            continue
        cands = [m for m in re.findall(pattern, log)
                 if not (exclude and m == exclude)]
        if cands:
            return cands[-1]
        if 'failed to request quick Tunnel' in log:
            return None
    return None

!pkill -f 'cloudflared tunnel' || true
open('/content/tunnel.log', 'w').close()
subprocess.Popen('nohup cloudflared tunnel --url http://localhost:8000 '
                 '> /content/tunnel.log 2>&1 &', shell=True)
url = None
for attempt in range(1, 4):
    print(f'cloudflared attempt {attempt}/3...')
    url = _wait_url('/content/tunnel.log',
                    r'https://[\w-]+\.trycloudflare\.com',
                    exclude='https://api.trycloudflare.com')
    if url:
        break
    subprocess.run("pkill -f 'cloudflared tunnel' || true", shell=True)
    time.sleep(5)

if not url:
    print('cloudflared blocked — falling back to localtunnel (no account needed)...')
    subprocess.run('npm install -g localtunnel', shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run("pkill -f 'lt --port 8000' || true", shell=True)
    open('/content/lt.log', 'w').close()
    subprocess.Popen('nohup lt --port 8000 > /content/lt.log 2>&1 &', shell=True)
    url = _wait_url('/content/lt.log', r'https://[\w-]+\.loca\.lt')

print('=' * 70)
print('PASTE THIS URL INTO THE DASHBOARD (GPU → Default Gpu → Colab):')
print(url or 'NO TUNNEL — check tunnel.log / lt.log, then re-run this cell')
print('=' * 70)

# ---- 5. keep-alive: heartbeat every 60s so the runtime stays awake ----
# Every dashboard poll also hits the server (request logs), and this steady
# output counts as activity. THIS CELL NEVER EXITS — that is intentional.
import json as _json
from datetime import datetime, timezone
from urllib.request import urlopen as _urlopen

while True:
    ts = datetime.now(timezone.utc).strftime('%H:%M:%S')
    try:
        health = _json.load(_urlopen('http://localhost:8000/health', timeout=10))
        if health.get('model_loaded'):
            status = 'READY — paste the tunnel URL in the dashboard'
        else:
            lp = health.get('load_progress', {}) or {}
            detail = (lp.get('detail') or '').split('/')[-1]
            status = f"loading: {lp.get('phase', '?')} {lp.get('pct', 0)}% {detail}".strip()
    except Exception as e:
        status = f'server not up yet ({e})'
    try:
        vram = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=memory.used,memory.total',
             '--format=csv,noheader,nounits'],
            text=True, timeout=10).strip()
    except Exception:
        vram = 'gpu query failed'
    print(f'[keep-alive {ts}] {status} | vram {vram} MB', flush=True)
    time.sleep(60)



## Anti-idle (recommended)

Press `Ctrl+Shift+I` → **Console** and paste this — it clicks Colab's Connect
button every 60 s so the tab never looks idle:

```js
setInterval(() => {
  const b = document.querySelector('#top-toolbar colab-connect-button');
  if (b) b.click();
}, 60000);
```

## Troubleshooting

- Server log: `!tail -n 30 /content/gpu_server.log`
- Ready check: `!curl -s localhost:8000/ready`
- Tunnel logs: `tunnel.log` (cloudflared) / `lt.log` (localtunnel fallback)
